![fa1](fa1.png)

In [20]:
import torch

# Q, K, V, output are tensors on the GPU
def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor,
          N: int, d_model: int, h: int):
    dk = d_model // h 
    q = Q.view(N, h, dk).transpose(0, 1)
    k = K.view(N, h, dk).transpose(0, 1)
    v = V.view(N, h, dk).transpose(0, 1)
    o = torch.zeros_like(q)
    q_blk_size, k_blk_size = min(16, N), min(16, N) 
    qblks = torch.split(q, q_blk_size, dim=1)
    kblks = torch.split(k, k_blk_size, dim=1)
    vblks = torch.split(v, k_blk_size, dim=1)
    oblks = list(torch.split(o, q_blk_size, dim=1))
    l = torch.zeros(*q.shape[:-1], 1, device=q.device, dtype=q.dtype)
    m = torch.full((*q.shape[:-1], 1), -1e10, device=q.device, dtype=q.dtype)
    lblks = list(torch.split(l, q_blk_size, dim=1))
    mblks = list(torch.split(m, q_blk_size, dim=1))
    scale = dk ** -0.5
    for j in range(len(kblks)):
        kj, vj = kblks[j], vblks[j]
        for i in range(len(qblks)):
            qi, oi, li, mi = qblks[i], oblks[i], lblks[i], mblks[i]
            x = qi @ kj.transpose(-1, -2) * scale 
            mblk, _ = torch.max(x, dim=-1, keepdim=True)
            pij = (x - mblk).exp()
            lblk = pij.sum(dim=-1, keepdim=True)
            pijvj = pij @ vj
            mi_new = torch.maximum(mblk, mi)
            i2new = (mi - mi_new).exp()
            b2new = (mblk - mi_new).exp()
            li_new = li * i2new + lblk * b2new 
            oblks[i] = oi * li / li_new * i2new + pijvj * b2new / li_new
            lblks[i], mblks[i] = li_new, mi_new 
    o = torch.cat(oblks, dim=1).transpose(0, 1).reshape(N, d_model)
    output.copy_(o)

In [21]:
import torch
ninf, eps = -1e10, 1e-10
qlen, klen, qblk, kblk = 6, 6, 3, 3
tr = qlen // qblk # tile row
tc = klen // kblk # tile col
batch, head, dk = 1, 1, 4
q = torch.randn(batch, head, qlen, dk)
k = torch.randn(batch, head, klen, dk)
v = torch.randn(batch, head, klen, dk)
o = torch.zeros_like(q)
l = torch.zeros(q.shape[:-1])[..., None]
m = torch.full(q.shape[:-1], ninf)[..., None]

qblks = torch.split(q, qblk, dim=2)
kblks = torch.split(k, kblk, dim=2)
vblks = torch.split(v, kblk, dim=2)
oblks = list(torch.split(o, qblk, dim=2))
lblks = list(torch.split(l, qblk, dim=2))
mblks = list(torch.split(m, qblk, dim=2))

for j in range(tc):
    kj = kblks[j]
    vj = vblks[j]
    for i in range(tr):
        qi = qblks[i]

        oi = oblks[i]
        li = lblks[i]
        mi = mblks[i]

        # x = torch.einsum('...id,...jd->...ij', qi, kj)
        x = qi @ kj.transpose(-2, -1)
        mblk, _ = torch.max(x, dim=-1, keepdim=True)
        pij = (x - mblk).exp()
        lblkij = torch.sum(pij, dim=-1, keepdim=True) + eps

        # pijvj = torch.einsum('...ij,...jd->...id', pij, vj)
        pijvj = pij @ vj

        mi_new = torch.maximum(mblk, mi)
        i2new = (mi - mi_new).exp()
        b2new = (mblk - mi_new).exp()
        li_new = li * i2new + lblkij * b2new
        oblks[i] = oi * li / li_new * i2new + pijvj * b2new / li_new
        lblks[i], mblks[i] = li_new, mi_new
        print(f'Q{i}K{j}: ')
        print(oblks[i])
o = torch.cat(oblks, dim=2)
l = torch.cat(lblks, dim=2)
m = torch.cat(mblks, dim=2)

Q0K0: 
tensor([[[[-0.2767, -0.1862, -0.4605,  0.2924],
          [ 0.6591, -1.3940,  0.6177, -0.6550],
          [ 0.5860, -1.3402,  0.5605, -0.5591]]]])
Q1K0: 
tensor([[[[ 0.6278, -1.4202,  0.6261, -0.5874],
          [-0.1714, -0.0859, -0.4965,  0.0586],
          [-0.4871, -0.5924, -0.2515,  0.8705]]]])
Q0K1: 
tensor([[[[-0.2524,  0.0487, -0.2957,  0.0111],
          [ 0.3758, -0.5142,  0.4200, -0.4805],
          [ 0.0939, -0.2929,  0.2108, -0.3696]]]])
Q1K1: 
tensor([[[[ 0.1294, -0.2375,  0.2485, -0.3579],
          [-0.0597,  0.4404,  0.0190, -0.4885],
          [-0.4502, -0.2022, -0.1419,  0.3583]]]])
